# Prepare PAD dataset for VLM training

Convert PAD-UFES-20 into the canonical JSONL format used by this project. **This is the primary dataset of the main study.**

- **Source:** `data/datasets/PAD/` — `metadata.csv` + `images/*.png` (zips are extracted if still present)
- **Output:** `data/processed/pad/clinical_context/`
- **Splits:** none on disk — 80/10/10 grouped by (`patient_id`, `lesion_id`), stratified by `diagnostic`
- **Canonical adapter:** `pad_prompt_and_label` — gold is canonical `code` only (`{"code": "<CODE>"}`); source diagnosis maps to that code; malignancy is derived later if a mapping exists
- Check duplicates / lesion overlap before treating the test split as held-out evaluation


## 1. Setup


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "src").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "src").exists():
    raise FileNotFoundError(f"Could not find repo root (src/) from {Path.cwd()}")

sys.path.insert(0, str(ROOT / "src"))

from vlm_ft.data.canonical import image_rel_path, pad_image_id, pad_prompt_and_label
from vlm_ft.data.prepare import (
    count_existing_images,
    load_pad,
    preview_processed,
    print_source_eda,
    rel_to_root,
    require_source,
    rows_to_samples,
    split_by_group,
    write_processed_dataset,
)

CURRENT_SOURCES = "PAD, ISIC18, HC, Derm1M, MILK10K"

PAD_ROOT = ROOT / "data/datasets/PAD"
OUT_DIR = ROOT / "data/processed/pad/clinical_context"
require_source(PAD_ROOT, expected=CURRENT_SOURCES)


## 2. Load


In [ ]:
df = load_pad(PAD_ROOT)
print(df.shape)
df.head()


## 3. EDA


In [ ]:
ok, total = count_existing_images(df, "image_rel", PAD_ROOT)
print(f"images on disk: {ok}/{total}")
print("images per lesion:\n", df.groupby("lesion_id").size().value_counts().sort_index().to_string())

splits = split_by_group(df, group_col=["patient_id", "lesion_id"], label_col="diagnostic")
eda = pd.concat(splits.values(), ignore_index=True)
print_source_eda(
    eda,
    label_col="diagnostic",
    metadata_cols=[
        "age", "gender", "fitspatrick", "smoke", "drink", "region",
        "diameter_1", "diameter_2", "itch", "grew", "hurt", "changed", "bleed", "elevation", "biopsed",
    ],
)


## 4. Convert to canonical JSONL


In [ ]:
PAD_ROOT_REL = rel_to_root(PAD_ROOT, ROOT)
samples = {
    split: rows_to_samples(
        frame,
        PAD_ROOT,
        prompt_and_label=pad_prompt_and_label,
        image_rel=lambda row: image_rel_path(pad_image_id(row)),
    )
    for split, frame in splits.items()
}
write_processed_dataset(
    name="pad/clinical_context",
    out_dir=OUT_DIR,
    split_samples=samples,
    image_root_rel=PAD_ROOT_REL,
)


## 5. Validate and preview


In [ ]:
preview_processed(OUT_DIR, "pad/clinical_context")
